In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
city_df_geo = pd.read_csv("data/city_safety/city_safety_2023_geo.csv")
country_df = pd.read_csv("data/crime-rate-by-country-2024.csv")

In [3]:
city_df = city_df_geo.copy()

# Clean country_norm on both sides
city_df['country'] = city_df['country'].astype(str).str.strip().str.lower()
country_df['country'] = country_df['country'].astype(str).str.strip().str.lower()

city_df['country_norm'] = city_df['country']
country_df['country_norm'] = country_df['country']

# Map state/province codes to countries
state_to_country = {
    'md': 'united states', 'tn': 'united states', 'mi': 'united states',
    'nm': 'united states', 'mo': 'united states', 'la': 'united states',
    'ca': 'united states', 'wi': 'united states', 'il': 'united states',
    'pa': 'united states', 'oh': 'united states', 'tx': 'united states',
    'ga': 'united states', 'ak': 'united states', 'dc': 'united states',
    'in': 'united states', 'ky': 'united states', 'fl': 'united states',
    'mn': 'united states', 'nv': 'united states', 'or': 'united states',
    'az': 'united states', 'wa': 'united states', 'ny': 'united states',
    'hi': 'united states', 'nc': 'united states', 'co': 'united states',
    'ma': 'united states', 'id': 'united states', 'ut': 'united states',
    'ab': 'canada', 'bc': 'canada',
}

name_fix = {
    'hong kong (china)': 'hong kong',
}

city_df['country_norm'] = (
    city_df['country_norm']
    .replace(state_to_country)
    .replace(name_fix)
)

In [5]:
# Standardize column names from the raw ta_data table
country_df = country_df.rename(columns={
    "Country": "country",
    "crimeRateByCountry_crimeIndex": "crime_index",
    "CrimeRate_OverallCriminalityScoreGOCI": "overall_criminality_score",
    "CrimeRate_CriminalMarketsScore": "criminal_markets_score",
    "CrimeRate_CriminalActorsScore": "criminal_actors_score",
    "CrimeRate_ResilienceScore": "resilience_score",
    "CrimeRateSafetyIndex": "safety_index",
})

country_df.columns.tolist()

['country',
 'crime_index',
 'overall_criminality_score',
 'criminal_markets_score',
 'criminal_actors_score',
 'resilience_score',
 'safety_index',
 'country_norm']

In [6]:
country_feats = [
    'country_norm',
    'crime_index',
    'overall_criminality_score',
    'criminal_markets_score',
    'criminal_actors_score',
    'resilience_score',
    'safety_index',
]

city_master = city_df.merge(
    country_df[country_feats],
    on='country_norm',
    how='left',
    suffixes=('', '_country')
)

print(city_master['crime_index_country'].isna().mean())

0.11057692307692307


In [7]:
missing_countries = (
    city_master.loc[city_master['crime_index_country'].isna(), 'country_norm']
    .value_counts()
)
print(missing_countries)

canada                37
colombia               3
thailand               3
dominican republic     1
moldova                1
georgia                1
Name: country_norm, dtype: int64


In [8]:
problem_list = [
    'canada', 'colombia', 'thailand',
    'dominican republic', 'moldova', 'georgia'
]

city_problem = (
    city_master
    .loc[city_master['country_norm'].isin(problem_list)
         & city_master['crime_index_country'].isna(),
         ['city', 'country', 'country_norm']]
    .sort_values(['country_norm', 'city'])
)

city_problem.head(80)

,city,country,country_norm
117,Brampton,canada,canada
337,Burlington,canada,canada
283,Calgary,canada,canada
387,Coquitlam,canada,canada
203,Edmonton,canada,canada
296,Guelph,canada,canada
243,Halifax,canada,canada
111,Hamilton,canada,canada
123,Kamloops,canada,canada
69,Kelowna,canada,canada


In [9]:
country_problem = country_df[
    country_df['country_norm'].isin(problem_list)
][['country', 'country_norm', 'crime_index']]

country_problem

,country,country_norm,crime_index
19,thailand,thailand,NaN
27,colombia,colombia,NaN
37,canada,canada,NaN
84,dominican republic,dominican republic,NaN
130,georgia,georgia,NaN
137,moldova,moldova,NaN


In [10]:
print(city_master.columns.tolist())

['Rank', 'city', 'country', 'crime_index', 'safety_index', 'year', 'city_norm', 'country_norm', 'lat', 'lon', 'crime_index_country', 'overall_criminality_score', 'criminal_markets_score', 'criminal_actors_score', 'resilience_score', 'safety_index_country']
